# About Project


I've mainly used nanochat repo https://github.com/karpathy/nanochat as it's recommended for a good and hackable implementation of GPT-2.



Because of that, I'm relying on scripts in the repo which are called from this notebook.



# Notable files used and modified from nanochat:

## Dataset

`nanochat.dataset`: I've redone this completely to load my own dataset.


The dataset is a mix of good sentences from devel.tsv and leipzig 1M news english dataset

`scripts.devel_test_train`: this splits devel.tsv into test and train sets

## Tokenizer

`scripts.tok_train` and `scripts.tok_eval`: remain almost unchanged, changed mainly hyperparameters to downscale tokenizer

## Training

`base_train`: main training script, almost unchanged. Changed mainly hyperparameters of the model to reduce the model size.

***After 10 minutes of training i've achieved 78% accuracy.***

`devel_fine_tune`: my own fine tuning script on the pair of sentences from devel.tsv datasaet.

***After another 10 minutes of training i've achieved 97% accuracy.***

## Evaluation

`eval_devel`, `eval_test`: my evaluation scripts









## Setup

In [1]:
import os

!git clone https://github.com/Depresivna-ryza/gpt-from-scratch.git

os.chdir('gpt-from-scratch')

print('Repository cloned and directory changed to gpt-from-scratch')

Cloning into 'gpt-from-scratch'...
remote: Enumerating objects: 1790, done.
remote: Counting objects: 100% (1790/1790), done.
remote: Compressing objects: 100% (626/626), done.
remote: Total 1790 (delta 1149), reused 1788 (delta 1147), pack-reused 0 (from 0)
Receiving objects: 100% (1790/1790), 1.91 MiB | 8.17 MiB/s, done.
Resolving deltas: 100% (1149/1149), done.
Repository cloned and directory changed to gpt-from-scratch


### Install Dependencies

In [2]:
!ls -la

total 724
drwxr-xr-x 10 root root   4096 May 20 20:30 .
drwxr-xr-x  1 root root   4096 May 20 20:30 ..
drwxr-xr-x  3 root root   4096 May 20 20:30 .claude
drwxr-xr-x  2 root root   4096 May 20 20:30 dev
drwxr-xr-x  8 root root   4096 May 20 20:30 .git
-rw-r--r--  1 root root    116 May 20 20:30 .gitignore
-rw-r--r--  1 root root   1072 May 20 20:30 LICENSE
drwxr-xr-x  2 root root   4096 May 20 20:30 nanochat
-rw-r--r--  1 root root   1366 May 20 20:30 pyproject.toml
-rw-r--r--  1 root root      5 May 20 20:30 .python-version
-rw-r--r--  1 root root  16740 May 20 20:30 README.md
drwxr-xr-x  2 root root   4096 May 20 20:30 runs
drwxr-xr-x  2 root root   4096 May 20 20:30 scripts
drwxr-xr-x  2 root root   4096 May 20 20:30 tasks
drwxr-xr-x  2 root root   4096 May 20 20:30 tests
-rw-r--r--  1 root root 662768 May 20 20:30 uv.lock


In [3]:
!uv sync --extra gpu

Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: .venv
Resolved 130 packages in 12ms
Prepared 81 packages in 1m 41s
Installed 81 packages in 629ms
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.12.15
 + aiosignal==1.4.0
 + annotated-types==0.7.0
 + anyio==4.10.0
 + async-timeout==5.0.1
 + attrs==25.3.0
 + certifi==2025.8.3
 + charset-normalizer==3.4.3
 + click==8.2.1
 + datasets==4.0.0
 + dill==0.3.8
 + exceptiongroup==1.3.0
 + fastapi==0.117.1
 + filelock==3.19.1
 + frozenlist==1.7.0
 + fsspec==2025.3.0
 + gitdb==4.0.12
 + gitpython==3.1.45
 + h11==0.16.0
 + hf-xet==1.1.9
 + huggingface-hub==0.34.4
 + idna==3.10
 + jinja2==3.1.6
 + kernels==0.11.7
 + markupsafe==3.0.2
 + mpmath==1.3.0
 + multidict==6.6.4
 + multiprocess==0.70.16
 + networkx==3.4.2
 + numpy==1.26.4
 + nvidia-cublas-cu12==12.8.4.1
 + nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cufft-c

In [4]:
import subprocess

def prepare_training_data():
    commands = [
        ['uv', 'run', '-m', 'nanochat.dataset'],
        ['uv', 'run', '-m', 'scripts.devel_test_train'],
        ['uv', 'run', '-m', 'scripts.tok_train'],
        ['uv', 'run', '-m', 'scripts.tok_eval']
    ]

    for i, cmd in enumerate(commands):
        print(f'\nRunning command: {' '.join(cmd)}...')
        try:
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in process.stdout:
                print(line, end='')
            process.wait()
            if process.returncode != 0:
                print(f"Error: Command {' '.join(cmd)} failed with exit code {process.returncode}")
                raise subprocess.CalledProcessError(process.returncode, cmd)
        except subprocess.CalledProcessError as e:
            print(f"An error occurred during {' '.join(cmd)}: {e}")
            break

    print('\nAll data preparation steps completed.')

In [5]:
prepare_training_data()


Running command: uv run -m nanochat.dataset...
Uninstalled 1 package in 157ms
Installed 1 package in 192ms
Created data directory at: data/datasets
Leipzig sentences downloaded and extracted successfully!
devel.tsv downloaded successfully!
eval.tsv downloaded successfully!
Leipzig corpus detected successfully!
Sample raw line: '1\t$100,000 will go towards the renewal of the existing trail at Spadonis Reserve.\n'
Devel TSV file detected successfully!
Sample raw line: 'Who should Derek hug after shocking Richard?\tWho should Derek hug Richard after shocking?\n'
Sample part 1 (grammatical): 'Who should Derek hug after shocking Richard?'
Sample part 2 (ungrammatical): 'Who should Derek hug Richard after shocking?'
Eval TSV file detected successfully!
Sample raw line: 'Who could Denise watch after insulting Dan?\tWho could Denise watch Dan after insulting?\n'
Sample part 1 : 'Who could Denise watch after insulting Dan?'
Sample part 2 : 'Who could Denise watch Dan after insulting?'

Checkin

In [6]:
def train_model():
    commands = [
        ['uv', 'run', '-m', 'scripts.base_train', '--num-iterations', '300', '--sample-every', '-1'],
        ['uv', 'run', '-m', 'scripts.devel_fine_tune', '--max-steps', '300', '--eval-every', '100', '--test-pairs-path', 'data/datasets/devel_val_split.tsv'],
    ]

    for i, cmd in enumerate(commands):
        print(f'\nRunning command: {' '.join(cmd)}...')
        try:
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in process.stdout:
                print(line, end='')
            process.wait()
            if process.returncode != 0:
                print(f"Error: Command {' '.join(cmd)} failed with exit code {process.returncode}")
                raise subprocess.CalledProcessError(process.returncode, cmd)
        except subprocess.CalledProcessError as e:
            print(f"An error occurred during {' '.join(cmd)}: {e}")
            break

    print('\nTraining Compelte.')

In [7]:
train_model()


Running command: uv run -m scripts.base_train --num-iterations 300 --sample-every -1...
2026-05-20 20:35:16,351 - nanochat.common - INFO - Distributed world size: 1
2026-05-20 20:35:16,352 - nanochat.common - WARNING - Peak flops undefined for: Tesla T4, MFU will show as 0%
W0520 20:35:31.745000 8537 .venv/lib/python3.10/site-packages/torch/_inductor/utils.py:1613] [0/0] Not enough SMs to use max_autotune_gemm mode

                                                       █████                █████
                                                      ░░███                ░░███
     ████████    ██████   ████████    ██████   ██████  ░███████    ██████  ███████
    ░░███░░███  ░░░░░███ ░░███░░███  ███░░███ ███░░███ ░███░░███  ░░░░░███░░░███░
     ░███ ░███   ███████  ░███ ░███ ░███ ░███░███ ░░░  ░███ ░███   ███████  ░███
     ░███ ░███  ███░░███  ░███ ░███ ░███ ░███░███  ███ ░███ ░███  ███░░███  ░███ ███
     ████ █████░░████████ ████ █████░░██████ ░░██████  ████ █████░░███████  ░░█████
 

In [8]:
def evaluate_model(filename):
    commands = [
        ['uv', 'run', '-m', 'scripts.eval_devel', '--max-examples', '2000', '--model-tag', 'd4', '--devel-path', 'data/datasets/devel_val_split.tsv'],
        ['uv', 'run', '-m', 'scripts.eval_devel', '--max-examples', '2000', '--model-tag', 'fine_tuned', '--devel-path', 'data/datasets/devel_val_split.tsv'],
        ['uv', 'run', '-m', 'scripts.eval_test','--model-tag', 'fine_tuned' ,'--test-path', filename],
    ]

    for i, cmd in enumerate(commands):
        print(f'\nRunning command: {' '.join(cmd)}...')
        try:
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in process.stdout:
                print(line, end='')
            process.wait()
            if process.returncode != 0:
                print(f"Error: Command {' '.join(cmd)} failed with exit code {process.returncode}")
                raise subprocess.CalledProcessError(process.returncode, cmd)
        except subprocess.CalledProcessError as e:
            print(f"An error occurred during {' '.join(cmd)}: {e}")
            break

    print('\nEvaluation Compelte.')

In [9]:
filename = 'data/datasets/eval-input.tsv'

evaluate_model(filename)


Running command: uv run -m scripts.eval_devel --max-examples 2000 --model-tag d4 --devel-path data/datasets/devel_val_split.tsv...
2026-05-20 20:59:21,057 - nanochat.common - INFO - Distributed world size: 1
2026-05-20 20:59:21,058 - nanochat.checkpoint_manager - INFO - Loading model from /content/gpt-from-scratch/data/base_checkpoints/d4 with step 300
2026-05-20 20:59:21,285 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 64, 'vocab_size': 8192, 'n_layer': 4, 'n_head': 2, 'n_kv_head': 2, 'n_embd': 128, 'window_pattern': 'L'}
Autodetected device type: cuda
Device: cuda (cuda)
Loading model with tag 'd4'...
✓ Model loaded
✓ Tokenizer loaded
Loaded 670 examples from data/datasets/devel_val_split.tsv

Evaluating: 100%|██████████| 670/670 [00:06<00:00, 104.93it/s]

Results on 670 examples from data/datasets/devel_val_split.tsv:
Accuracy: 532/670 = 79.40%


Running command: uv run -m scripts.eval_devel --max-examples 2000 --model-tag fine_tuned --devel-p

In [11]:
# Validate output format


with open(filename, "r") as f:
    lines = f.readlines()

with open("data/datasets/eval_output.output", "r") as f:
    outputs = f.readlines()


for (i, (line, output)) in enumerate(zip(lines, outputs)):
    in_1, in_2 = line.strip().split("\t")
    out = output.strip()


    if out != in_1 and out != in_2:
        print(f"Warning: output does not match either input for line {i+1}")
        print(f"Input 1: {in_1}")
        print(f"Input 2: {in_2}")
        print(f"Output : {out}")
        print("-" * 40)
        break
else:
    print("All outputs match either input")




All outputs match either input
